# ChatApp Python Front-End: Impostor Demo

This notebook shows how to use the current `chatapp` facade for an Impostor-style game.

The demo keeps everything self-contained by running against the FastAPI app through `TestClient`, so you do not need to start a separate server. It also uses a scripted LLM client to demonstrate where runtime-driven player behavior fits into the facade.

In [3]:
from __future__ import annotations



from pathlib import Path

from tempfile import TemporaryDirectory

from pprint import pprint



from fastapi.testclient import TestClient



import chatapp

import api.websockets as websocket_module

from chatapp import TestClientChatGateway

from chatapp.options import pause_group_chat, read_messages, resume_group_chat, send_messages

from db.session import create_connection, get_db, init_db

from main import app

from simulation.runtimes.llm import LLMPlayerRuntimeFactory, resolve_llm_provider_config



workspace = TemporaryDirectory()

database_path = Path(workspace.name) / "impostor-demo.db"

init_db(database_path)



def testing_session_local():

    return create_connection(database_path)



def override_get_db():

    db = testing_session_local()

    try:

        yield db

    finally:

        db.close()



app.dependency_overrides[get_db] = override_get_db

original_session_local = websocket_module.SessionLocal

websocket_module.SessionLocal = testing_session_local



client_manager = TestClient(app)

client = client_manager.__enter__()

gateway = TestClientChatGateway(client)

server = chatapp.init_server(gateway=gateway)

provider_config = resolve_llm_provider_config()



def conversation_contents(conversation, viewer=None):

    messages = conversation.list_messages(viewer=viewer)

    return [message["content"] for message in messages]



print(f"Demo database: {database_path}")

pprint({"provider": provider_config.provider, "model": provider_config.model, "base_url": provider_config.base_url})


Demo database: /var/folders/cm/sn2bpgln3zdc0lh8s2384hr40000gn/T/tmpu9t06j1r/impostor-demo.db
{'base_url': 'https://api.pinference.ai/api/v1',
 'model': 'meta-llama/llama-3.3-70b-instruct',
 'provider': 'primeintellect'}


In [4]:
admin = server.add_member(

    name="Admin",

    runtime_type="human",

    member_type="admin",

    functionalities=[send_messages, read_messages, pause_group_chat, resume_group_chat],

)



players = [

    server.add_member(

        name=name,

        runtime_type="llm",

        member_type="user_regular",

        functionalities=[send_messages, read_messages],

        config={"simulation_runtime": "llm"},

    )

    for name in ["Claudia", "Diego", "Marta", "Luis"]

]



runtime_factory = LLMPlayerRuntimeFactory.from_environment(provider_config.provider)



for player in players:

    player.attach_runtime(

        runtime_factory.create(

            player_name=player.display_name,

            member_id=player.id,

            gateway=server.gateway,

        )

    )



pprint(

    {

        "provider": provider_config.provider,

        "model": provider_config.model,

        "admin": admin.payload,

        "players": [player.payload for player in players],

    }

)


{'admin': {'capabilities': {'create_direct_conversations': True,
                            'create_group_conversations': True,
                            'leave_conversations': True,
                            'manage_memberships': True,
                            'pause_group_messages': True,
                            'read_conversations': True,
                            'send_messages': True},
           'config': None,
           'display_name': 'Admin',
           'id': '93c30c89-f24e-4ea9-ab42-100511a1ac14',
           'member_type': 'admin',
           'type': 'human'},
 'model': 'meta-llama/llama-3.3-70b-instruct',
 'players': [{'capabilities': {'create_direct_conversations': True,
                               'create_group_conversations': False,
                               'leave_conversations': True,
                               'manage_memberships': False,
                               'pause_group_messages': False,
                               'read_conver

In [5]:
group_chat = server.open_session(title="Impostor", owner=admin, members=players)
private_chats = {
    player.display_name: admin.start_direct_chat(
        title=f"Admin and {player.display_name}",
        members=[player],
    )
    for player in players
}

admin.send_message(group_chat, "Rules of the Game: answer with Ready if you are ready to play.")

for player in players:
    ready_message = player.runtime.decide_ready(
        group_conversation_id=group_chat.id,
        ready_text="Ready",
    )
    player.send_message(group_chat, ready_message)

admin.send_message(group_chat, "Group chat is temporarily paused while private words are assigned.")
admin.pause_group_chat(group_chat, "Group chat is temporarily paused while private words are assigned.")

assignments = {
    "Claudia": "apple",
    "Diego": "apple",
    "Marta": "apple",
    "Luis": "pear",
}

for player in players:
    admin.send_message(
        private_chats[player.display_name],
        f"Your secret word is: {assignments[player.display_name]}",
    )

admin.resume_group_chat(group_chat)
admin.send_message(group_chat, "Round 1 begins now.")

print(group_chat.payload)

{'id': 'df2ac42b-3132-4652-9ee7-cfa2a293ab68', 'type': 'group', 'title': 'Impostor', 'participant_ids': ['c36632d3-ffee-45e3-94fa-6fa46970b460', '93c30c89-f24e-4ea9-ab42-100511a1ac14', '9bb296f7-6ef3-4eff-b554-48346f2c3ac7', '9a722fff-95a5-4d48-90c1-f07de820fe4a', '74571add-2937-42e2-a3c5-3a3ec77a74ab'], 'messages_paused': False, 'message_pause_notice': None}


In [6]:
for player in players:
    clue = player.runtime.decide_clue(
        group_conversation_id=group_chat.id,
        private_conversation_id=private_chats[player.display_name].id,
    )
    player.send_message(group_chat, f"{player.display_name} clue: {clue}")

for player in players:
    admin.send_message(
        private_chats[player.display_name],
        "Cast your vote for the player you think has a different word.",
    )

votes = {}
for player in players:
    vote = player.runtime.decide_vote(
        group_conversation_id=group_chat.id,
        private_conversation_id=private_chats[player.display_name].id,
        player_names=[candidate.display_name for candidate in players],
    )
    votes[player.display_name] = vote
    player.send_message(private_chats[player.display_name], vote)

vote_totals = {player.display_name: 0 for player in players}
for vote in votes.values():
    vote_totals[vote] += 1

eliminated_player = max(vote_totals, key=lambda name: (vote_totals[name], name == "Luis", name))
admin.send_message(group_chat, "Vote results: " + ", ".join(f"{name}={vote_totals[name]}" for name in sorted(vote_totals)))
admin.send_message(group_chat, f"Impostor eliminated: {eliminated_player}.")

pprint(votes)
pprint(vote_totals)

{'Claudia': 'Marta', 'Diego': 'Marta', 'Luis': 'Marta', 'Marta': 'Luis'}
{'Claudia': 0, 'Diego': 0, 'Luis': 1, 'Marta': 3}


In [7]:
print("Group transcript")

for content in conversation_contents(group_chat):

    print("-", content)



print("\nWhat Claudia can read in her private chat")

for content in conversation_contents(private_chats["Claudia"], viewer=players[0]):

    print("-", content)



print("\nRuntime provider")

pprint({"provider": provider_config.provider, "model": provider_config.model})


Group transcript
- Rules of the Game: answer with Ready if you are ready to play.
- Ready
- Ready
- Ready
- Ready
- Group chat is temporarily paused while private words are assigned.
- Round 1 begins now.
- Claudia clue: fruit
- Diego clue: juicy
- Marta clue: crunchy
- Luis clue: sweet
- Vote results: Claudia=0, Diego=0, Luis=1, Marta=3
- Impostor eliminated: Marta.

What Claudia can read in her private chat
- Your secret word is: apple
- Cast your vote for the player you think has a different word.
- Marta

Runtime provider
{'model': 'meta-llama/llama-3.3-70b-instruct', 'provider': 'primeintellect'}


## Notes

- `chatapp.init_server(...)` is the entry point for the Python facade.
- Members get their chat powers through `functionalities=[...]`, which map onto the backend capability model.
- The runtime is attached separately from capabilities. In this notebook it is a scripted LLM client, but the same slot can be backed by a real provider-driven runtime.
- The scenario code only talks to the chat facade objects: `ChatServer`, `ChatMember`, and `ChatConversation`.

In [8]:
server.close()
client_manager.__exit__(None, None, None)
app.dependency_overrides.clear()
websocket_module.SessionLocal = original_session_local
runtime_factory.close()
workspace.cleanup()
print("Cleaned up notebook resources.")

Cleaned up notebook resources.
